# Layer V ET Identity Gene Analysis — Combined DNA Methylation
Combines **CGN** and **CHN** DNA-methylation h5ad files into a single AnnData,
then identifies genes that define Layer V ET identity through stepwise comparison:
- **Non-Neuron vs Layer V ET** -> Neuron-identity genes
- **Inhibitory-Neuron vs Layer V ET** -> Excitatory-neuron-identity genes
- **Upper Layer vs Layer V ET** -> Deep-layer-identity genes
- **Other Deep Layer vs Layer V ET** -> Layer V ET-specific genes

> **Note on DNA methylation data:**  
> Methylation fraction values (0-1) are used directly as a gene-activity proxy.
> Higher gene-body methylation generally correlates with *lower* expression.


In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path

sc.settings.verbosity = 1


/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: 

## 1. Load & Combine DNA Methylation Data

In [2]:
# -- EDIT THESE PATHS --------------------------------------------------------
CGN_PATH = "/home/nakagawa/datasets/h5ad/DNA_Methylation_CGN.h5ad"
CHN_PATH = "/home/nakagawa/datasets/h5ad/DNA_Methylation_CHN.h5ad"
OUT_DIR  = Path("/home/nakagawa/datasets/LayerV_ET_results_methylation")
# ----------------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading CGN ...")
adata_cgn = sc.read_h5ad(CGN_PATH)
adata_cgn.obs["methylation_context"] = "CGN"
print(" ", adata_cgn)

print("Loading CHN ...")
adata_chn = sc.read_h5ad(CHN_PATH)
adata_chn.obs["methylation_context"] = "CHN"
print(" ", adata_chn)

# Keep only genes present in BOTH datasets
shared_genes = adata_cgn.var_names.intersection(adata_chn.var_names)
print(f"\nShared genes: {len(shared_genes)}  "
      f"(CGN-only: {len(adata_cgn.var_names)-len(shared_genes)}, "
      f"CHN-only: {len(adata_chn.var_names)-len(shared_genes)})")

adata_cgn = adata_cgn[:, shared_genes].copy()
adata_chn = adata_chn[:, shared_genes].copy()

# Concatenate along cells (axis=0)
adata = ad.concat(
    [adata_cgn, adata_chn],
    axis=0,
    join="inner",        # keep only obs columns shared by both datasets
    label="dataset",     # new column adata.obs['dataset']
    keys=["CGN", "CHN"],
    index_unique="-",    # makes barcodes unique: barcode-CGN / barcode-CHN
)

# Copy var metadata from CGN (both are now identical after alignment)
adata.var = adata_cgn.var.copy()

print("\n=== Combined AnnData ===")
print(adata)
print(f"\nCells per context:")
print(adata.obs['dataset'].value_counts())


Loading CGN ...
  AnnData object with n_obs × n_vars = 9876 × 51574
    obs: 'FullCellID', 'CCC_Rate', 'CG_Rate', 'CG_RateAdj', 'CH_Rate', 'CH_RateAdj', 'FinalReads', 'InputReads', 'MappedReads', 'Region', 'index_name', 'uid', 'BamFilteringRate', 'MappingRate', 'Pos96', 'Plate', 'Col96', 'Row96', 'Col384', 'Row384', 'FACS_Date', 'Slice', 'PassFilter', 'pca_0', 'pca_1', 'umap_0', 'umap_1', 'tsne_0', 'tsne_1', 'MajorCluster', 'SubCluster', 'BICCN_class_label', 'BICCN_subclass_label', 'BICCN_cluster_label', 'BICCN_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'donor_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'methylation_context'
    var: 'chrom', 'start', 'end', 'chr_s

In [3]:
# Inspect raw dataset structures before proceeding
print("=== CGN obs columns ===")
print(adata_cgn.obs.columns.tolist())
print(adata_cgn.obs.head(3))
print("\n=== CGN var ===")
print(adata_cgn.var.head(3))
print("\n=== CHN obs columns ===")
print(adata_chn.obs.columns.tolist())
print(adata_chn.obs.head(3))
print("\n=== CHN var ===")
print(adata_chn.var.head(3))


=== CGN obs columns ===
['FullCellID', 'CCC_Rate', 'CG_Rate', 'CG_RateAdj', 'CH_Rate', 'CH_RateAdj', 'FinalReads', 'InputReads', 'MappedReads', 'Region', 'index_name', 'uid', 'BamFilteringRate', 'MappingRate', 'Pos96', 'Plate', 'Col96', 'Row96', 'Col384', 'Row384', 'FACS_Date', 'Slice', 'PassFilter', 'pca_0', 'pca_1', 'umap_0', 'umap_1', 'tsne_0', 'tsne_1', 'MajorCluster', 'SubCluster', 'BICCN_class_label', 'BICCN_subclass_label', 'BICCN_cluster_label', 'BICCN_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'donor_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'methylation_context']
                                          FullCellID  CCC_Rate  CG_Rate  \
0  180508_CEMBA_

## 2. Find the Cell Type Column & List All Cell Types

In [8]:
# Show all metadata columns so you can identify the right cell type column
print("Available metadata columns:")
print(adata.obs.columns.tolist())

# Try common cell-type column names for methylation data
for candidate in ['cell_type', 'BICCN_subclass_label', 'BICCN_cluster_label']:
    if candidate in adata.obs.columns:
        print(f"Found column '{candidate}':")
        print(adata.obs[candidate].value_counts())


Available metadata columns:
['FullCellID', 'CCC_Rate', 'CG_Rate', 'CG_RateAdj', 'CH_Rate', 'CH_RateAdj', 'FinalReads', 'InputReads', 'MappedReads', 'Region', 'index_name', 'uid', 'BamFilteringRate', 'MappingRate', 'Pos96', 'Plate', 'Col96', 'Row96', 'Col384', 'Row384', 'FACS_Date', 'Slice', 'PassFilter', 'pca_0', 'pca_1', 'umap_0', 'umap_1', 'tsne_0', 'tsne_1', 'MajorCluster', 'SubCluster', 'BICCN_class_label', 'BICCN_subclass_label', 'BICCN_cluster_label', 'BICCN_ontology_term_id', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'donor_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'methylation_context', 'dataset']
Found column 'cell_type':
cell_type
glutamatergic neuron              15304


In [9]:
# -- EDIT: set to the column name that contains cell type labels ------------
# Adjust after checking the output above (common names for methylation data:
# 'subclass_label', 'cell_type', 'CellType', 'cluster')
CELLTYPE_COL = "BICCN_cluster_label"   # <-- change this if needed
# ----------------------------------------------------------------------------

cell_types = sorted(adata.obs[CELLTYPE_COL].unique().tolist())
print(f"\nFound {len(cell_types)} cell types in '{CELLTYPE_COL}':\n")
for i, ct in enumerate(cell_types):
    n = (adata.obs[CELLTYPE_COL] == ct).sum()
    print(f"  [{i:02d}] {ct}  (n={n})")



Found 43 cell types in 'BICCN_cluster_label':

  [00] CGE-VipNdnf_Lamp5-Ndnf  (n=192)
  [01] CGE-VipNdnf_Lamp5-Unc5d  (n=392)
  [02] CGE-VipNdnf_Pex5l  (n=86)
  [03] CGE-VipNdnf_Vip-Ano4  (n=222)
  [04] CGE-VipNdnf_Vip-Prr16  (n=142)
  [05] CGE-VipNdnf_Vip-Sorcs1  (n=100)
  [06] CGE-VipNdnf_Vip-Unc5b  (n=116)
  [07] L23-IT-Cux2  (n=3930)
  [08] L4-IT-Rorb_Rorb-Cpne4  (n=752)
  [09] L4-IT-Rorb_Rorb-Ryr3  (n=384)
  [10] L4-IT-Rorb_Rorb-Tenm2  (n=1996)
  [11] L5-IT-Deptor  (n=1796)
  [12] L5-PT-Bcl6  (n=708)
  [13] L6-CT-Foxp2_Foxp2-Glra2  (n=896)
  [14] L6-CT-Foxp2_Foxp2-Kcnh5  (n=968)
  [15] L6-CT-Foxp2_Foxp2-Wscd1  (n=652)
  [16] L6-CT-Foxp2_Hcrtr2  (n=278)
  [17] L6-CT-Foxp2_Spon1  (n=366)
  [18] L6-IT-Sulf1_Cables1  (n=716)
  [19] L6-IT-Sulf1_Meis2  (n=532)
  [20] L6-IT-Sulf1_Pcdh15  (n=46)
  [21] L6-IT-Sulf1_Sulf1  (n=452)
  [22] L6-NP-Tshz2  (n=550)
  [23] L6b-Galnt10  (n=282)
  [24] MGE-Pvalb_Chrna7  (n=472)
  [25] MGE-Pvalb_Cnih3  (n=248)
  [26] MGE-Pvalb_Man2a1  (n=184)
  [27] 

## 3. Define Layer V ET and Classify Other Cell Types
After seeing the list above, fill in the three categories below.

In [15]:
# ── EDIT THESE after reading cell type list above ────────────────────────────

LAYER_V_ET = "L5-PT-Bcl6"

NON_NEURON_TYPES = [
    "NonN_Astro-Slc1a2",
    "NonN_Endo-Nxn",
    "NonN_SMC-Myo1b",
    "NonN_VLMC-Slc6a13",
]

UPPER_LAYER_TYPES = [
    "L23-IT-Cux2",
    "L4-IT-Rorb_Rorb-Tenm2",
    "L4-IT-Rorb_Rorb-Cpne4",
    "L4-IT-Rorb_Rorb-Ryr3",
]

OTHER_DEEP_LAYER_TYPES = {
    "L5_IT":  ["L5-IT-Deptor"],
    "L6_NP":  ["L6-NP-Tshz2"],
    "L6_CT":  ["L6-CT-Foxp2_Foxp2-Kcnh5", "L6-CT-Foxp2_Foxp2-Glra2",
               "L6-CT-Foxp2_Foxp2-Wscd1", "L6-CT-Foxp2_Hcrtr2", "L6-CT-Foxp2_Spon1"],
    "L6_IT":  ["L6-IT-Sulf1_Cables1", "L6-IT-Sulf1_Meis2",
               "L6-IT-Sulf1_Sulf1", "L6-IT-Sulf1_Pcdh15"],
    # L6b is debatable — optionally add: "L6b": ["L6b-Galnt10"]
}

INTERNEURON_TYPES = [
    "MGE-Pvalb_Chrna7",
    "MGE-Pvalb_Cnih3",
    "MGE-Pvalb_Man2a1",
    "MGE-Pvalb_Unc5b",
    "MGE-Sst_Gfra2",
    "MGE-Sst_Daam2",
    "MGE-Sst_Srrm4",
    "MGE-Sst_Prep",
    "MGE-Sst_Kcnip4",
    "MGE-Sst_Whrn",
    "MGE-Sst_Chodl",
    "CGE-VipNdnf_Lamp5-Unc5d",
    "CGE-VipNdnf_Lamp5-Ndnf",
    "CGE-VipNdnf_Vip-Ano4",
    "CGE-VipNdnf_Vip-Prr16",
    "CGE-VipNdnf_Vip-Unc5b",
    "CGE-VipNdnf_Vip-Sorcs1",
    "CGE-VipNdnf_Pex5l",
]

# Your target
TARGET_TYPE = "L5 ET"

# ── Thresholds ───────────────────────────────────────────────────────────────
LOG2FC_THRESH  = 1.0    # absolute log2 fold change cutoff (2-fold)
PVAL_THRESH    = 0.05   # adjusted p-value (FDR)
# ─────────────────────────────────────────────────────────────────────────────

print(f"Target: {LAYER_V_ET}")
print(f"Non-neuron types  : {NON_NEURON_TYPES}")
print(f"Upper layer types : {UPPER_LAYER_TYPES}")
print(f"Other deep layer  : {OTHER_DEEP_LAYER_TYPES}")
print(f"GABAergic inhibitory neurons : {INTERNEURON_TYPES}")

Target: L5-PT-Bcl6
Non-neuron types  : ['NonN_Astro-Slc1a2', 'NonN_Endo-Nxn', 'NonN_SMC-Myo1b', 'NonN_VLMC-Slc6a13']
Upper layer types : ['L23-IT-Cux2', 'L4-IT-Rorb_Rorb-Tenm2', 'L4-IT-Rorb_Rorb-Cpne4', 'L4-IT-Rorb_Rorb-Ryr3']
Other deep layer  : {'L5_IT': ['L5-IT-Deptor'], 'L6_NP': ['L6-NP-Tshz2'], 'L6_CT': ['L6-CT-Foxp2_Foxp2-Kcnh5', 'L6-CT-Foxp2_Foxp2-Glra2', 'L6-CT-Foxp2_Foxp2-Wscd1', 'L6-CT-Foxp2_Hcrtr2', 'L6-CT-Foxp2_Spon1'], 'L6_IT': ['L6-IT-Sulf1_Cables1', 'L6-IT-Sulf1_Meis2', 'L6-IT-Sulf1_Sulf1', 'L6-IT-Sulf1_Pcdh15']}
GABAergic inhibitory neurons : ['MGE-Pvalb_Chrna7', 'MGE-Pvalb_Cnih3', 'MGE-Pvalb_Man2a1', 'MGE-Pvalb_Unc5b', 'MGE-Sst_Gfra2', 'MGE-Sst_Daam2', 'MGE-Sst_Srrm4', 'MGE-Sst_Prep', 'MGE-Sst_Kcnip4', 'MGE-Sst_Whrn', 'MGE-Sst_Chodl', 'CGE-VipNdnf_Lamp5-Unc5d', 'CGE-VipNdnf_Lamp5-Ndnf', 'CGE-VipNdnf_Vip-Ano4', 'CGE-VipNdnf_Vip-Prr16', 'CGE-VipNdnf_Vip-Unc5b', 'CGE-VipNdnf_Vip-Sorcs1', 'CGE-VipNdnf_Pex5l']


## 4. Preprocessing

In [17]:
# -- Methylation-aware preprocessing ----------------------------------------
#
# DNA methylation fraction values are typically already in [0, 1] (or [0, 100]).
# We do NOT library-size-normalize or log1p transform as with scRNA-seq.
# Steps:
#   1. Convert % -> fraction if values are stored as 0-100
#   2. Fill NaN with the per-gene mean (missing CpG coverage)
#   3. X is left as-is; Wilcoxon rank-sum is rank-based and robust to
#      the bounded [0,1] distribution of methylation values.
# ---------------------------------------------------------------------------

import scipy.sparse as sp

adata_use = adata.copy()

# Convert to dense float32
if sp.issparse(adata_use.X):
    X = adata_use.X.toarray().astype(np.float32)
else:
    X = np.array(adata_use.X, dtype=np.float32)

# Convert % to fraction if needed
if np.nanmax(X) > 1.5:
    print("Values > 1 detected -- dividing by 100 to convert % to fraction")
    X = X / 100.0

# Impute NaN with per-gene mean
nan_count = int(np.isnan(X).sum())
if nan_count > 0:
    print(f"Imputing {nan_count} NaN values with gene means")
    gene_means = np.nanmean(X, axis=0)
    nan_mask = np.isnan(X)
    X[nan_mask] = np.take(gene_means, np.where(nan_mask)[1])

adata_use.X = X
adata_use.obs[CELLTYPE_COL] = adata.obs[CELLTYPE_COL]

print("Preprocessing complete.")
print(f"  X shape : {adata_use.X.shape}")
print(f"  X range : [{np.nanmin(adata_use.X):.3f}, {np.nanmax(adata_use.X):.3f}]")
print(f"  NaN remaining: {int(np.isnan(adata_use.X).sum())}")

# OPTIONAL: run DE per context separately
# adata_use_cgn = adata_use[adata_use.obs['dataset'] == 'CGN'].copy()
# adata_use_chn = adata_use[adata_use.obs['dataset'] == 'CHN'].copy()


Values > 1 detected -- dividing by 100 to convert % to fraction
Preprocessing complete.
  X shape : (19752, 51573)
  X range : [0.000, 1.155]
  NaN remaining: 0


## 5. Run DE Analysis: Each Cell Type vs Layer V ET

In [19]:
def run_de(adata_use, celltype_col, group_a, group_b, log2fc_thresh, pval_thresh):
    """
    Run Wilcoxon rank-sum DE between group_a vs group_b.
    Positive log2FC = upregulated in group_a relative to group_b (Layer V ET).
    """
    mask = adata_use.obs[celltype_col].isin([group_a, group_b])
    sub  = adata_use[mask].copy()
    sub.obs["group"] = sub.obs[celltype_col].astype(str)

    sc.tl.rank_genes_groups(
        sub,
        groupby="group",
        groups=[group_a],
        reference=group_b,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
    )

    result = sc.get.rank_genes_groups_df(sub, group=group_a)
    result = result.rename(columns={
        "names"       : "gene",
        "logfoldchanges": "log2FC",
        "pvals_adj"   : "padj",
        "pvals"       : "pval",
        "scores"      : "score",
    })

    result["-log10padj"] = -np.log10(result["padj"].clip(lower=1e-300))
    result["comparison"] = f"{group_a}_vs_LayerVET"

    sig = result[
        (result["padj"] < pval_thresh) &
        (result["log2FC"].abs() >= log2fc_thresh)
    ].copy()

    sig["direction"] = np.where(sig["log2FC"] > 0, "UP_in_other", "UP_in_LayerVET")

    return result, sig


all_comparisons   = {}   # full results
all_sig           = {}   # significant only

all_cell_types = NON_NEURON_TYPES + UPPER_LAYER_TYPES + INTERNEURON_TYPES + [
    ct for clusters in OTHER_DEEP_LAYER_TYPES.values() for ct in clusters
]

for ct in all_cell_types:
    if ct not in adata_use.obs[CELLTYPE_COL].values:
        print(f"[SKIP] '{ct}' not found in data")
        continue
    print(f"[DE]  {ct} vs {LAYER_V_ET} ...", end=" ")
    full, sig = run_de(adata_use, CELLTYPE_COL, ct, LAYER_V_ET, LOG2FC_THRESH, PVAL_THRESH)
    all_comparisons[ct] = full
    all_sig[ct]         = sig
    print(f"{len(sig)} significant genes (UP_other={( sig.direction=='UP_in_other').sum()}, UP_LayerVET={(sig.direction=='UP_in_LayerVET').sum()})")

[DE]  NonN_Astro-Slc1a2 vs L5-PT-Bcl6 ... 428 significant genes (UP_other=397, UP_LayerVET=31)
[DE]  NonN_Endo-Nxn vs L5-PT-Bcl6 ... 416 significant genes (UP_other=375, UP_LayerVET=41)
[DE]  NonN_SMC-Myo1b vs L5-PT-Bcl6 ... 648 significant genes (UP_other=581, UP_LayerVET=67)
[DE]  NonN_VLMC-Slc6a13 vs L5-PT-Bcl6 ... 574 significant genes (UP_other=524, UP_LayerVET=50)
[DE]  L23-IT-Cux2 vs L5-PT-Bcl6 ... 69 significant genes (UP_other=47, UP_LayerVET=22)
[DE]  L4-IT-Rorb_Rorb-Tenm2 vs L5-PT-Bcl6 ... 90 significant genes (UP_other=70, UP_LayerVET=20)
[DE]  L4-IT-Rorb_Rorb-Cpne4 vs L5-PT-Bcl6 ... 69 significant genes (UP_other=52, UP_LayerVET=17)
[DE]  L4-IT-Rorb_Rorb-Ryr3 vs L5-PT-Bcl6 ... 76 significant genes (UP_other=50, UP_LayerVET=26)
[DE]  MGE-Pvalb_Chrna7 vs L5-PT-Bcl6 ... 235 significant genes (UP_other=130, UP_LayerVET=105)
[DE]  MGE-Pvalb_Cnih3 vs L5-PT-Bcl6 ... 256 significant genes (UP_other=158, UP_LayerVET=98)
[DE]  MGE-Pvalb_Man2a1 vs L5-PT-Bcl6 ... 257 significant genes

## 6. Save Individual Comparison CSVs

In [20]:
for ct, sig in all_sig.items():
    safe_name = ct.replace("/", "_").replace(" ", "_")
    path = OUT_DIR / f"{safe_name}_vs_LayerVET.csv"
    sig.sort_values("log2FC", ascending=False).to_csv(path, index=False)
    print(f"Saved: {path.name}")

Saved: NonN_Astro-Slc1a2_vs_LayerVET.csv
Saved: NonN_Endo-Nxn_vs_LayerVET.csv
Saved: NonN_SMC-Myo1b_vs_LayerVET.csv
Saved: NonN_VLMC-Slc6a13_vs_LayerVET.csv
Saved: L23-IT-Cux2_vs_LayerVET.csv
Saved: L4-IT-Rorb_Rorb-Tenm2_vs_LayerVET.csv
Saved: L4-IT-Rorb_Rorb-Cpne4_vs_LayerVET.csv
Saved: L4-IT-Rorb_Rorb-Ryr3_vs_LayerVET.csv
Saved: MGE-Pvalb_Chrna7_vs_LayerVET.csv
Saved: MGE-Pvalb_Cnih3_vs_LayerVET.csv
Saved: MGE-Pvalb_Man2a1_vs_LayerVET.csv
Saved: MGE-Pvalb_Unc5b_vs_LayerVET.csv
Saved: MGE-Sst_Gfra2_vs_LayerVET.csv
Saved: MGE-Sst_Daam2_vs_LayerVET.csv
Saved: MGE-Sst_Srrm4_vs_LayerVET.csv
Saved: MGE-Sst_Prep_vs_LayerVET.csv
Saved: MGE-Sst_Kcnip4_vs_LayerVET.csv
Saved: MGE-Sst_Whrn_vs_LayerVET.csv
Saved: MGE-Sst_Chodl_vs_LayerVET.csv
Saved: CGE-VipNdnf_Lamp5-Unc5d_vs_LayerVET.csv
Saved: CGE-VipNdnf_Lamp5-Ndnf_vs_LayerVET.csv
Saved: CGE-VipNdnf_Vip-Ano4_vs_LayerVET.csv
Saved: CGE-VipNdnf_Vip-Prr16_vs_LayerVET.csv
Saved: CGE-VipNdnf_Vip-Unc5b_vs_LayerVET.csv
Saved: CGE-VipNdnf_Vip-Sorcs1_v

## 7. Stepwise Candidate Narrowing

Logic:
- **Neuron-identity genes** = UP in Layer V ET vs ANY non-neuron type
- **Exitory-Neuron-identity genes** = above AND UP in Layer V ET vs ANY interneurons type
- **Deep-layer-identity genes** = above AND UP in Layer V ET vs ANY upper layer type
- **Layer V ET-specific genes** = above AND UP in Layer V ET vs ALL other deep layer types

In [24]:
def get_layerVET_up_genes(all_sig, cell_types):
    """Genes consistently UP in Layer V ET (direction == UP_in_LayerVET) across given comparisons."""
    sets = []
    for ct in cell_types:
        if ct in all_sig:
            up = set(all_sig[ct][all_sig[ct]["direction"] == "UP_in_LayerVET"]["gene"])
            sets.append(up)
    if not sets:
        return set()
    # Gene must appear in AT LEAST ONE comparison to be included at each step
    return set.union(*sets)


# Step 1: UP vs non-neurons
neuron_identity_genes = get_layerVET_up_genes(all_sig, NON_NEURON_TYPES)
print(f"Step 1 — Neuron-identity genes: {len(neuron_identity_genes)}")

# Step 2: UP vs interneurons (independent filter, not chained)
interneuron_up = get_layerVET_up_genes(all_sig, INTERNEURON_TYPES)
excitatory_neuron_identity_genes = neuron_identity_genes & interneuron_up
print(f"Step 2 — Excitatory-neuron-identity genes: {len(excitatory_neuron_identity_genes)}")

# Step 3: use Step 1 base (not Step 2) AND upper layer — less restrictive
upper_layer_up = get_layerVET_up_genes(all_sig, UPPER_LAYER_TYPES)
deep_layer_identity_genes = neuron_identity_genes & upper_layer_up  # <-- branch from Step 1
print(f"Step 3 — Deep-layer-identity genes: {len(deep_layer_identity_genes)}")

print("Step 3 genes:", deep_layer_identity_genes)
print("Step 2 genes:", excitatory_neuron_identity_genes)
print("Overlap (Step2 & Step3):", excitatory_neuron_identity_genes & deep_layer_identity_genes)
print("\nWhich deep layer group kills them?")
for group_name, group_up in zip(OTHER_DEEP_LAYER_TYPES.keys(), deep_layer_group_sets):
    surviving = excitatory_neuron_identity_genes & deep_layer_identity_genes & group_up
    print(f"  After {group_name}: {len(surviving)} genes remaining")

# Step 4: intersection of Step 2 AND Step 3 AND deep layer groups
# gene must be excitatory AND deep AND beat all deep layer groups
deep_layer_group_sets = []
for group_name, clusters in OTHER_DEEP_LAYER_TYPES.items():
    group_up = set()
    for ct in clusters:
        if ct in all_sig:
            up = set(all_sig[ct][all_sig[ct]["direction"] == "UP_in_LayerVET"]["gene"])
            group_up |= up
    if group_up:
        deep_layer_group_sets.append(group_up)
        print(f"  {group_name}: {len(group_up)} UP_in_LayerVET genes")

if deep_layer_group_sets:
    consistently_up_vs_deep = set.intersection(*deep_layer_group_sets)
    layerVET_specific_genes = excitatory_neuron_identity_genes & deep_layer_identity_genes & consistently_up_vs_deep
else:
    layerVET_specific_genes = set()

print(f"Step 4 — Layer V ET-specific genes: {len(layerVET_specific_genes)}")

Step 1 — Neuron-identity genes: 149
Step 2 — Excitatory-neuron-identity genes: 38
Step 3 — Deep-layer-identity genes: 6
Step 3 genes: {'ENSMUSG00000029705', 'ENSMUSG00000000247', 'ENSMUSG00000104369', 'ENSMUSG00000086220', 'ENSMUSG00000028266', 'ENSMUSG00000030839'}
Step 2 genes: {'ENSMUSG00000029705', 'ENSMUSG00000027562', 'ENSMUSG00000030310', 'ENSMUSG00000029778', 'ENSMUSG00000055197', 'ENSMUSG00000039313', 'ENSMUSG00000086220', 'ENSMUSG00000069769', 'ENSMUSG00000037242', 'ENSMUSG00000026805', 'ENSMUSG00000022018', 'ENSMUSG00000021379', 'ENSMUSG00000036478', 'ENSMUSG00000087027', 'ENSMUSG00000104369', 'ENSMUSG00000029335', 'ENSMUSG00000032531', 'ENSMUSG00000037892', 'ENSMUSG00000026220', 'ENSMUSG00000021994', 'ENSMUSG00000087694', 'ENSMUSG00000055435', 'ENSMUSG00000108489', 'ENSMUSG00000029178', 'ENSMUSG00000097929', 'ENSMUSG00000026932', 'ENSMUSG00000086496', 'ENSMUSG00000013275', 'ENSMUSG00000028128', 'ENSMUSG00000102424', 'ENSMUSG00000062078', 'ENSMUSG00000024427', 'ENSMUSG000000

## 8. Build Summary Tables & Save

In [25]:
# -- Gene symbol mapping ----------------------------------------------------
# Methylation h5ad may store gene names in var_names directly,
# or in a column like 'feature_name', 'gene_name', 'gene_symbol'.
print("var columns:", adata.var.columns.tolist())
print(adata.var.head(3))

SYMBOL_COL = None
for candidate in ['feature_name', 'gene_name', 'gene_symbol', 'symbol']:
    if candidate in adata.var.columns:
        SYMBOL_COL = candidate
        break

if SYMBOL_COL:
    id_to_symbol = adata.var[SYMBOL_COL].to_dict()
    print(f"Using '{SYMBOL_COL}' for gene symbols")
else:
    id_to_symbol = {g: g for g in adata.var_names}
    print("No symbol column found -- using var_names as gene symbols")

print(f"Loaded {len(id_to_symbol)} gene symbols -- example: {list(id_to_symbol.items())[:3]}")

# -- Build summary tables per gene set ----------------------------------------
def build_summary(gene_set, all_comparisons, label):
    rows = []
    for gene in sorted(gene_set):
        row = {
            "gene":        gene,
            "gene_symbol": id_to_symbol.get(gene, gene),
            "category":    label,
        }
        for ct, df in all_comparisons.items():
            match = df[df["gene"] == gene]
            if not match.empty:
                row[f"{ct}__log2FC"]    = round(match["log2FC"].values[0], 3)
                row[f"{ct}__log10padj"] = round(match["-log10padj"].values[0], 3)
                row[f"{ct}__padj"]      = match["padj"].values[0]
        rows.append(row)
    return pd.DataFrame(rows)

df_neuron   = build_summary(neuron_identity_genes,          all_comparisons, "neuron_identity")
df_exitory  = build_summary(exitory_neuron_identity_genes,  all_comparisons, "exitory_neuron_identity")
df_deep     = build_summary(deep_layer_identity_genes,      all_comparisons, "deep_layer_identity")
df_specific = build_summary(layerVET_specific_genes,        all_comparisons, "LayerVET_specific")

df_neuron.to_csv(  OUT_DIR / "neuron_identity_genes.csv",          index=False)
df_exitory.to_csv( OUT_DIR / "exitory_neuron_identity_genes.csv",  index=False)
df_deep.to_csv(    OUT_DIR / "deep_layer_identity_genes.csv",      index=False)
df_specific.to_csv(OUT_DIR / "LayerVET_specific_genes.csv",        index=False)

print(f"Saved to {OUT_DIR}/")
print(f"  neuron_identity_genes.csv              : {len(df_neuron)} genes")
print(f"  exitory_neuron_identity_genes.csv      : {len(df_exitory)} genes")
print(f"  deep_layer_identity_genes.csv          : {len(df_deep)} genes")
print(f"  LayerVET_specific_genes.csv            : {len(df_specific)} genes")


var columns: ['chrom', 'start', 'end', 'chr_start_stop', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
                    chrom      start        end            chr_start_stop  \
ENSMUSG00000050732   chr6   72385222   72390702    chr6_72385222_72390702   
ENSMUSG00000101535   chr1  128128409  128128888  chr1_128128409_128128888   
ENSMUSG00000101517  chr10   95746076   95797235   chr10_95746076_95797235   

                    feature_is_filtered   feature_name feature_reference  \
ENSMUSG00000050732                False          Vamp8   NCBITaxon:10090   
ENSMUSG00000101535                False        Gm29383   NCBITaxon:10090   
ENSMUSG00000101517                False  4732465J04Rik   NCBITaxon:10090   

                   feature_biotype feature_length          feature_type  
ENSMUSG00000050732            gene           1448        protein_coding  
ENSMUSG00000101535            gene            480  processed_pseudog

## 9. Preview Results

In [26]:
print("\n=== TOP 20 Layer V ET-SPECIFIC GENES ===")
if not df_specific.empty:
    log2fc_cols = [c for c in df_specific.columns if c.endswith("__log2FC")]
    df_specific["mean_log2FC"] = df_specific[log2fc_cols].mean(axis=1)
    display(df_specific[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]  # ← added gene_symbol
            .sort_values("mean_log2FC", ascending=False)
            .head(20)
            .reset_index(drop=True))
else:
    print("No Layer V ET-specific genes found — consider relaxing thresholds (LOG2FC_THRESH / PVAL_THRESH)")

print("\n=== TOP 20 DEEP LAYER IDENTITY GENES ===")
if not df_deep.empty:
    log2fc_cols = [c for c in df_deep.columns if c.endswith("__log2FC")]
    df_deep["mean_log2FC"] = df_deep[log2fc_cols].mean(axis=1)
    display(df_deep[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]  # ← added gene_symbol
            .sort_values("mean_log2FC", ascending=False)
            .head(20)
            .reset_index(drop=True))


=== TOP 20 Layer V ET-SPECIFIC GENES ===
No Layer V ET-specific genes found — consider relaxing thresholds (LOG2FC_THRESH / PVAL_THRESH)

=== TOP 20 DEEP LAYER IDENTITY GENES ===


,gene,gene_symbol,mean_log2FC,NonN_Astro-Slc1a2__log2FC,NonN_Endo-Nxn__log2FC,NonN_SMC-Myo1b__log2FC,NonN_VLMC-Slc6a13__log2FC,L23-IT-Cux2__log2FC,L4-IT-Rorb_Rorb-Tenm2__log2FC,L4-IT-Rorb_Rorb-Cpne4__log2FC,...,L6-NP-Tshz2__log2FC,L6-CT-Foxp2_Foxp2-Kcnh5__log2FC,L6-CT-Foxp2_Foxp2-Glra2__log2FC,L6-CT-Foxp2_Foxp2-Wscd1__log2FC,L6-CT-Foxp2_Hcrtr2__log2FC,L6-CT-Foxp2_Spon1__log2FC,L6-IT-Sulf1_Cables1__log2FC,L6-IT-Sulf1_Meis2__log2FC,L6-IT-Sulf1_Sulf1__log2FC,L6-IT-Sulf1_Pcdh15__log2FC
0,ENSMUSG00000028266,Lmo4,0.051757,0.484,-1.057,0.755,0.039,-0.511,-1.109,-1.179,...,0.048,-0.313,-0.229,-0.607,-0.789,-0.882,-1.003,-1.336,-0.753,-1.449
1,ENSMUSG00000000247,Lhx2,-0.360730,0.205,-1.139,-1.107,-1.148,-1.666,-1.334,-1.028,...,0.093,0.199,0.197,0.279,0.203,0.115,0.107,-0.026,0.210,-0.065
2,ENSMUSG00000029705,Cux1,-0.577460,-1.180,-1.033,-0.876,-1.017,-1.866,-1.266,-0.774,...,-0.707,-0.052,-0.017,-0.059,-0.117,-0.021,-0.092,-0.063,0.008,-0.696
3,ENSMUSG00000086220,Gm16599,-0.606054,-1.158,-0.952,-1.047,-1.068,-1.757,-1.266,-0.784,...,-0.815,-0.275,-0.213,-0.261,-0.217,-0.172,-0.089,-0.049,0.044,-0.616
4,ENSMUSG00000104369,Gm38082,-0.639513,-1.161,-1.150,-0.986,-0.796,-1.626,-1.225,-0.712,...,-0.777,-0.085,-0.013,-0.079,-0.247,-0.129,-0.243,-0.267,-0.020,-0.831
5,ENSMUSG00000030839,Sergef,-1.173108,-0.599,-1.012,-0.904,-0.967,-1.240,-0.891,-0.917,...,-0.905,-0.743,-0.691,-1.029,-0.833,-0.697,-1.091,-0.843,-0.802,-1.606


## 10. Threshold Sensitivity Check (Optional)
Run this if Step 3 returns too few or too many genes.

In [27]:
print(adata.var.columns.tolist())
print(adata.var.head())

['chrom', 'start', 'end', 'chr_start_stop', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
                    chrom      start        end             chr_start_stop  \
ENSMUSG00000050732   chr6   72385222   72390702     chr6_72385222_72390702   
ENSMUSG00000101535   chr1  128128409  128128888   chr1_128128409_128128888   
ENSMUSG00000101517  chr10   95746076   95797235    chr10_95746076_95797235   
ENSMUSG00000103873  chr12  114910010  114910442  chr12_114910010_114910442   
ENSMUSG00000099414  chr15   75326499   75326727    chr15_75326499_75326727   

                    feature_is_filtered   feature_name feature_reference  \
ENSMUSG00000050732                False          Vamp8   NCBITaxon:10090   
ENSMUSG00000101535                False        Gm29383   NCBITaxon:10090   
ENSMUSG00000101517                False  4732465J04Rik   NCBITaxon:10090   
ENSMUSG00000103873                False       Ighv1-38   NCBITaxon:100

In [28]:
print("Sensitivity check — gene counts at different thresholds:\n")
print(f"{'log2FC':>8}  {'padj':>6}  {'Neuron':>8}  {'DeepLayer':>10}  {'LayerVET_specific':>18}")

for lfc in [0.5, 1.0, 1.5, 2.0]:
    for pv in [0.1, 0.05, 0.01]:
        def get_up(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    up = set(df[(df["padj"] < pv) & (df["log2FC"] < -lfc)]["gene"])
                    sets.append(up)
            return set.union(*sets) if sets else set()

        def get_up_intersect(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    up = set(df[(df["padj"] < pv) & (df["log2FC"] < -lfc)]["gene"])
                    sets.append(up)
            return set.intersection(*sets) if sets else set()

        n  = len(get_up(NON_NEURON_TYPES))
        en = len(get_up(NON_NEURON_TYPES) & get_up(INTERNEURON_TYPES))
        ul = len(get_up(NON_NEURON_TYPES) & get_up(INTERNEURON_TYPES) & get_up(UPPER_LAYER_TYPES))
        sp = len(get_up(NON_NEURON_TYPES) & get_up(UPPER_LAYER_TYPES) & get_up_intersect(OTHER_DEEP_LAYER_TYPES))
        print(f"{lfc:>8.1f}  {pv:>6.2f}  {n:>8}  {ul:>10}  {sp:>18}")

Sensitivity check — gene counts at different thresholds:

  log2FC    padj    Neuron   DeepLayer   LayerVET_specific
     0.5    0.10      4680         149                   0
     0.5    0.05      4680         148                   0
     0.5    0.01      4669         147                   0
     1.0    0.10       149           4                   0
     1.0    0.05       149           4                   0
     1.0    0.01       149           4                   0
     1.5    0.10         2           0                   0
     1.5    0.05         2           0                   0
     1.5    0.01         2           0                   0
     2.0    0.10         0           0                   0
     2.0    0.05         0           0                   0
     2.0    0.01         0           0                   0
